In [1]:
import pandas as pd
import numpy as np

In [2]:

# Replace with your actual CSV file path
df = pd.read_csv("../data/Manufacturer_Data.csv")

In [3]:


# Delivery score (fast delivery better)
df["delivery_score"] = 1 / (df["Lead_time_days"] + 1)

# Quantity score (higher stock better)
df["quantity_score"] = df["Current_Stock"] / (df["Current_Stock"].max() + 1)

# Cost score (lower price better)
df["cost_score"] = 1 / (df["Unit_Price"] + 1)

# Quality score (fewer returns better)
df["quality_rate"] = 1 - (
    (df["Total_orders"] - df["Return_Units"]) /
    (df["Total_orders"] + 1)
)

# Response score (faster response better)
df["response_score"] = 1 - (df["Response_Time_in_hrs"] / 48)

# Clean invalid values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Clip values
df["response_score"] = df["response_score"].clip(0,1)
df["quality_rate"] = df["quality_rate"].clip(0,1)

In [4]:

# ------------------------------------------------
# RELIABILITY SCORE CALCULATION (YOUR FORMULA)
# ------------------------------------------------
df["reliability_score"] = (
    0.25 * df["delivery_score"] +
    0.15 * df["quantity_score"] +
    0.20 * df["cost_score"] +
    0.25 * df["quality_rate"] +
    0.15 * df["response_score"]
)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
df["product_text"] = (
    df["Product_sub_Category"] + " " + df["Product_Category"] + " " + df["Product_Name"] + " " + df["Product_Description"]
)

vectorizer = TfidfVectorizer(stop_words="english")
product_vectors = vectorizer.fit_transform(df["product_text"])

In [7]:
def recommend_supplier_by_search(
    search_query,
    required_qty,
    similarity_threshold=0.2,
    top_n=4
):
    # Vectorize search query
    search_vector = vectorizer.transform([search_query])

    # Compute similarity
    similarities = cosine_similarity(search_vector, product_vectors)[0]
    df["similarity_score"] = similarities

    # Filter relevant products
    filtered = df[
        (df["similarity_score"] >= similarity_threshold) &
        (df["Current_Stock"] >= required_qty)
    ]

    if filtered.empty:
        print("❌ No matching products found for your search")
        return

    # Rank suppliers
    ranked = filtered.sort_values(
        by=["reliability_score", "Unit_Price", "Lead_time_days"],
        ascending=[False, True, True]
    )

    # Take TOP 4
    top_recommended = ranked.head(top_n)

    print("\n🔍 Search:", search_query)
    print(f"✅ Top {len(top_recommended)} Recommended Suppliers:\n")

    for i, (_, row) in enumerate(top_recommended.iterrows(), start=1):
        print(
            f"{i}. Manufacturer: {row['Manufacturer_id']} | "
            f"Product: {row['Product_Name']} | "
            f"Price: ₹{row['Unit_Price']} | "
            f"Delivery: {row['Lead_time_days']} days | "
            f"Stock: {row['Current_Stock']} | "
            f"Reliability Score: {row['reliability_score']}"
        )


In [8]:
recommend_supplier_by_search(
    search_query="laptop",
    required_qty=4
)


🔍 Search: laptop
✅ Top 4 Recommended Suppliers:

1. Manufacturer: M9860 | Product: Ultrabook Laptop Model 185 | Price: ₹2389.36 | Delivery: 4.0 days | Stock: 4246.0 | Reliability Score: 0.5694158891708749
2. Manufacturer: M9860 | Product: Ultrabook Laptop Model 473 | Price: ₹563.37 | Delivery: 17.0 days | Stock: 4872.0 | Reliability Score: 0.5485047064439798
3. Manufacturer: M0213 | Product: Student Laptop Model 654 | Price: ₹705.88 | Delivery: 6.0 days | Stock: 4231.0 | Reliability Score: 0.542396829820135
4. Manufacturer: M4000 | Product: Ultrabook Laptop Model 853 | Price: ₹2165.35 | Delivery: 8.0 days | Stock: 4742.0 | Reliability Score: 0.5396322600860428
